In [1]:
import os
import cv2
import json
# from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import scipy
import cv2
from skimage.measure import regionprops, label, shannon_entropy
from skimage.transform import resize
from skimage.color import rgb2gray
from skimage import img_as_ubyte
from brisque import BRISQUE
from skvideo.measure import niqe
from pypiqe import piqe
from mahotas.features import zernike_moments
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms
import timm
import segmentation_models_pytorch as smp

# Patch imresize if missing
if not hasattr(scipy.misc, "imresize"):
    def imresize(arr, size, interp=None, mode=None):
        if isinstance(size, float):  # scale factor
            new_shape = (int(arr.shape[0] * size), int(arr.shape[1] * size))
        else:
            new_shape = size[:2]
        arr_resized = resize(arr, new_shape, order=3, mode="reflect", anti_aliasing=True)
        arr_resized = (arr_resized * 255).astype(np.uint8)
        return arr_resized
    scipy.misc.imresize = imresize

# Patch for deprecated NumPy aliases (for backward compatibility)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

In [2]:
# Hyperparameters and paths
DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/CUB_200_2011/'
NUM_PARTS = 12      # Including background
CLASSIFIER_NAME = "efficientnetv2_rw_m.agc_in1k"
SEGMENTATION_NAME = "deeplabv3plus"
SEGMENTATION_ENCODER = "resnext50_32x4d"
CLASSIFIER_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/CUB/deit_base_distilled_patch16_224.fb_in1k_timm_logs/checkpoints/best_model.pth"
SEGMENTATION_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/2_segmentation/CUB/deeplabv3plus/lightning_logs/version_2/checkpoints/epoch=199-step=13400.ckpt"
# SHAPE_FEATS_PATH = r"/home/c/choton/beemachine/codes/everyday_test/November_25/Nov12_25/shape_features/beemachine_small_2025_part_features.csv"
# FEATURE_SIZE = 937

# Model training hyperparameters
MODEL = "arch2_cub_red_effnet_v2m"
DEVICE_ID = 2
BATCH_SIZE = 64
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
LOG_DIR = f"./{MODEL}_logs/"
SEED = 42

# Seeding for consistant results
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# Create SIFT and ORB detectors once
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
bri_obj = BRISQUE(url=False)

def extract_base_features(mask):
    """Compute geometric, Zernike, Fourier, and texture shape descriptors from a binary mask."""
    
    features = ["area", "perimeter", "aspect_ratio", "extent", "solidity", "eccentricity", 
        "orientation", "circularity", "elongation", "compactness"]
    
    if mask is None or mask.sum() == 0:
        return {f: 0 for f in features}

    # --- Region properties ---
    # mask = mask.astype(np.uint8)
    labeled = label(mask)
    props = regionprops(labeled)
    if len(props) == 0:
        return {f: 0 for f in features}
    p = props[0]
    major_axis = p.major_axis_length
    minor_axis = p.minor_axis_length

    # ----- base shape features -----
    area = p.area
    perimeter = max(p.perimeter, 1e-6) # Ignoring too small perimeters
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else 0 # L_major / L_minor
    extent = p.extent
    solidity = p.solidity
    eccentricity = p.eccentricity
    orientation = p.orientation
    circularity = 4 * np.pi * area / (perimeter ** 2)
    elongation = 1 - (minor_axis / major_axis) if major_axis > 0 else 0
    # convexity = p.perimeter_convex / perimeter
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)

    # ----- Assemble features -----
    features_d = {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "orientation": orientation,
        "circularity": circularity,
        "elongation": elongation,
        "compactness": compactness
    }
    return features_d

def compute_sift_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY) # converts image into uint8 and mask as input
    keypoints, descriptors = sift.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 128), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_orb_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 32), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_hu_moments(mask):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.log(np.abs(hu) + 1e-12) # log-scale for stability
    return hu

def compute_zernike_moments(mask, degree=8):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    radius = min(mask.shape) // 2
    mask_norm = mask / mask.max() if mask.max() > 0 else mask
    zern = zernike_moments(mask_norm, radius=radius, degree=degree)
    return zern

# *** Updated fourier descriptors (Dec 4, 2025)
def compute_fourier_descriptors(mask, image=None, fourier_harmonics=20, visualize=False):
    if not isinstance(mask, np.ndarray): # Ensure proper mask format
        mask = mask.numpy().astype(np.uint8)
    # --- 2. Find largest contour (object part) ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        # Too few points for Fourier transform
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    
    # Translation invariance: center contour
    complex_contour = cnt[:,0,0] + 1j * cnt[:,0,1]
    fd = np.fft.fft(complex_contour)
    
    if visualize: # ** IMPORTANT: Visualization uses raw contour (so you can see the real shape), descriptors are centered.
        # Convert image if needed
        H, W = mask.shape
        if image is not None:
            if isinstance(image, torch.Tensor):
                image = image.detach().cpu().numpy().transpose(1, 2, 0)
            elif isinstance(image, Image.Image):
                image = np.array(image.convert('RGB'))
            elif image.dtype != np.uint8:  # NumPy float → uint8
                image = (image*255).astype(np.uint8)
            img_draw = image.copy()
        else:
            img_draw = np.zeros((H, W, 3), dtype=np.uint8)
        cv2.drawContours(img_draw, [cnt.astype(np.int32)], -1, (0, 255, 0), 2)

        fd_recon = fd.copy()
        keep = fourier_harmonics
        if 2 * keep < len(fd_recon):
            fd_recon[keep:-keep] = 0 # Safe zeroing
        else:
            fd_recon[keep:] = 0
        recon = np.fft.ifft(fd_recon)
        pts = np.column_stack((recon.real, recon.imag)).astype(np.int32)

        for i in range(len(pts)):
            cv2.line(img_draw, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (255, 0, 255), 1)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title("Shape Descriptors Overlay")
        plt.legend(
            handles=[
                Patch(facecolor='green', edgecolor='green'),
                Patch(facecolor='magenta', edgecolor='magenta')
            ],
            labels=["Contour", "Fourier Reconstruction"],
            loc='upper right'
        )
        plt.show()
    
    cnt_centered = complex_contour - np.mean(complex_contour)
    fd = np.fft.fft(cnt_centered)
    if len(fd) < 2 or np.abs(fd[1]) == 0:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)

    # Scale invariance: divide by first descriptor magnitude
    fd = fd / np.abs(fd[1])

    # Rotation invariance: use only magnitudes
    fd_normalized = np.abs(fd)

    # Keep only first N harmonics
    fd_truncated = fd_normalized[:fourier_harmonics]
    if len(fd_truncated) < fourier_harmonics:
        fd_truncated = np.concatenate([fd_truncated, np.full((fourier_harmonics - len(fd_truncated)), np.nan)])
    return fd_truncated

def extract_shape_features(image, mask):
    # Compute base features
    features = extract_base_features(mask)

    # Compute sift features
    sift_kp, sift_ds = compute_sift_features(image, mask)
    sift_sizes = [k.size for k in sift_kp]
    if sift_ds.shape[0] > 0:
        sift_mean_ds = np.nanmean(sift_ds, axis=0)
    else:
        sift_mean_ds = np.full(128, np.nan)
    sift_dict = {f'sift_ds{i+1}': sift_mean_ds[i] for i in range(len(sift_mean_ds))}
    sift_dict['sift_kp_n'] = len(sift_kp)
    sift_dict['sift_kp_size'] = np.max(sift_sizes) if sift_sizes else 0

    # Compute orb features
    orb_kp, orb_ds = compute_orb_features(image, mask)
    if orb_ds.shape[0] > 0:
        orb_mean_ds = np.nanmean(orb_ds, axis=0)
    else:
        orb_mean_ds = np.full(32, np.nan)
    orb_dict = {f'orb_ds{i+1}': orb_mean_ds[i] for i in range(len(orb_mean_ds))}
    orb_dict['orb_kp_n'] = len(orb_kp)

    # Compute hu moments
    hu_moments = compute_hu_moments(mask)
    hu_dict = {f"hu{i+1}": hu_moments[i] for i in range(len(hu_moments))}

    # Compute Zernike moments
    zern_moments = compute_zernike_moments(mask, degree=8)
    zern_dict = {f"zernike_{i+1}": zern_moments[i] for i in range(len(zern_moments))}

    # Compute fourier descriptors
    fourier_ds = compute_fourier_descriptors(mask, fourier_harmonics=20)
    fourier_dict = {f"fourier_{i+1}": fourier_ds[i] for i in range(len(fourier_ds))}

    features.update(sift_dict)
    features.update(orb_dict)
    features.update(hu_dict)
    features.update(zern_dict)
    features.update(fourier_dict)
    converted = {k: np.float32(v) for k, v in features.items()}
    return converted

def extract_visual_features(image, mask):
    # --- 1. Ensure binary uint8 mask ---
    if not isinstance(mask, np.ndarray):
        mask = mask.numpy().astype(np.uint8)
    # Convert image to numpy
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    elif isinstance(image, Image.Image):
        image = np.array(image.convert('RGB'))
    img_cropped = np.zeros_like(image)
    img_cropped[mask==1] = image[mask==1]
    # plt.imshow(img_cropped)

    # --- Brightness ---
    brightness = np.mean(img_cropped)

    # --- Contrast (standard deviation of luminance) ---
    gray = rgb2gray(img_cropped)
    contrast = np.std(gray)

    # --- Sharpness (variance of Laplacian) ---
    gray_8u = (gray * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(gray_8u, cv2.CV_64F).var()

    # --- Colorfulness (Hasler & Süsstrunk, 2003) ---
    (R, G, B) = cv2.split(img_cropped)
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Entropy (texture complexity) ---
    entropy = shannon_entropy(gray)

    # BRISQUE
    bri_obj = BRISQUE(url=False)
    brisque_score = bri_obj.score(img_cropped)

    # NIQE
    niqe_score = niqe(gray)

    # PIQE
    piqe_score, activityMask, noticeableArtifactMask, noiseMask = piqe(gray)

    # --- Aggregate descriptors ---
    descriptors = {
        "brightness": np.float32(brightness),
        "contrast": np.float32(contrast),
        "sharpness": np.float32(lap_var),
        "colorfulness": np.float32(colorfulness),
        "entropy": np.float32(entropy),
        "brisque": np.float32(brisque_score),
        "niqe": np.float32(niqe_score.item()),
        "piqe": np.float32(piqe_score)
    }
    return descriptors

def extract_combined_features(image, mask): 
    # ---- Convert once ----
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    if isinstance(image, Image.Image):
        image = np.asarray(image)
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy().astype(np.uint8)
    mask_u8 = mask.astype(np.uint8)
    image_f = (image - image.min()) / (image.max() - image.min() + 1e-8) # Linearly rescale to [0, 1] and avoid division by zero
    image_u8 = img_as_ubyte(image_f)

    combined_features = extract_shape_features(image_u8, mask_u8)
    vis_features = extract_visual_features(image_f, mask_u8)
    combined_features.update(vis_features)
    return combined_features

def extract_all_features(image, mask):
    """
    Generalized feature extraction for N body parts.
    mask: 2D array where each unique integer >0 is a body part ID.
    """

    # Get unique body-part IDs (excluding background 0)
    part_ids = sorted([pid for pid in np.unique(mask) if pid > 0])

    # Extract features for each part
    part_features = {}
    for pid in part_ids:
        part_mask = mask == pid
        feats = extract_combined_features(image, part_mask)
        part_features[pid] = feats

    # Full body features
    full_mask = mask > 0
    full_feats = extract_combined_features(image, full_mask)

    # Compute area sums
    areas = {pid: part_features[pid]["area"] for pid in part_ids}
    total_area = sum(areas.values()) + 1e-6

    # ----------- Ratio Features -----------
    # Pairwise ratios: area_i / area_j
    ratio_feats = {}
    for i in part_ids:
        for j in part_ids:
            if i != j:
                ratio_feats[f"area_ratio_part{i}_to_part{j}"] = areas[i] / (areas[j] + 1e-6)

    # Area-to-total ratios
    for pid in part_ids:
        ratio_feats[f"area_ratio_part{pid}_to_total"] = areas[pid] / total_area

    # ----------- Build Record -----------
    record = {}

    # Add each part's features
    for pid in part_ids:
        feats = part_features[pid]
        for k, v in feats.items():
            record[f"part{pid}_{k}"] = v

    # Add full-body features
    for k, v in full_feats.items():
        record[f"full_{k}"] = v

    # Add ratio features
    record.update(ratio_feats)
    return record

In [4]:
# Load classifier
def load_classifier(name, weights, num_classes, device_id):
    # Define the classification model and load weights
    model = timm.create_model(name, pretrained=False, num_classes=num_classes).to(f"cuda:{device_id}")
    state_dict_c = torch.load(weights, map_location=f"cuda:{device_id}")
    model.load_state_dict(state_dict_c) # Load the classifier state dict
    return model

# Load segmentation model
def load_segmentation(name, encoder_name, checkpoint_path):    
    # Define the segmentation model and load weights
    model = smp.create_model(
                name,
                encoder_name=encoder_name,
                in_channels = 3, # 3 for RGB images
                classes = 12
            ).to(f"cuda:{DEVICE_ID}")
    # checkpoint_path = r"/home/c/choton/beemachine/codes/everyday_test/oct1_25/segmentation_models/lightning_logs/fpn/lightning_logs/version_0/checkpoints/epoch=199-step=37400.ckpt"
    checkpoint = torch.load(checkpoint_path, map_location=rf"cuda:{DEVICE_ID}")

    # Sometimes Lightning adds "state_dict"
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

        # Remove "model." or "net." prefixes if present
        new_state_dict = {k.replace("model.", "").replace("net.", ""): v 
                        for k, v in state_dict.items() if k not in ["mean", "std"]}

        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(checkpoint)
    # model.to(f"cuda:{DEVICE_ID}")
    return model

In [5]:
# =========================
# 2. LOAD OFFICIAL SPLITS
# =========================

images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["img_id", "filepath"]
)

labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["img_id", "label"]
)

split_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["img_id", "is_train"]
)

# Merge all
df = images_df.merge(labels_df, on="img_id")
df = df.merge(split_df, on="img_id")

# Convert labels to 0-index
df["label"] = df["label"] - 1

# =========================
# 3. CREATE 75:15:10 SPLIT
# =========================

# First separate official test
official_train = df[df["is_train"] == 1]
official_test = df[df["is_train"] == 0]

# Combine all data
full_df = pd.concat([official_train, official_test])

# 75% train, 25% temp
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.25,
    stratify=full_df["label"],
    random_state=SEED
)

# Split temp into val (15%) and test (10%)
val_ratio = 0.15 / (0.15 + 0.10)

val_df, test_df = train_test_split(
    temp_df,
    test_size=(1 - val_ratio),
    stratify=temp_df["label"],
    random_state=SEED
)

print(f"Train: {len(train_df)}, Val:   {len(val_df)}, Test:  {len(test_df)}")
full_df

Train: 8841, Val:   1768, Test:  1179


,img_id,filepath,label,is_train
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
6,7,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
7,8,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
...,...,...,...,...
11779,11780,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11782,11783,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11784,11785,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11785,11786,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0


In [6]:
class CUBDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DATA_DIR, "images", row["filepath"])
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label, idx

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = CUBDataset(train_df, train_transform)
val_dataset = CUBDataset(val_df, val_transform)
test_dataset = CUBDataset(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

classes = set(labels_df['label'])
num_classes = len(classes)
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 200 | Train: 8841 | Val: 1768 | Test: 1179


In [8]:
# ======================================================
# Helper: Load shape features
# ======================================================
def load_shape_features(csv_path):
    df = pd.read_csv(csv_path) #.drop(columns=["label"])
    # df = df.fillna(df.mean())
    # labels = torch.tensor(df["label"].values, dtype=torch.long)
    # Drop non-feature columns: assuming CSV columns are ['image', 'label', ...features]
    return df

In [9]:
train_feats_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/dim_reduction/padc_cub_dim_search_logs/cub_reduced_train_GRP.csv"
val_feats_path   = r"/home/c/choton/beemachine/codes/AG_vision_2026/dim_reduction/padc_cub_dim_search_logs/cub_reduced_val_GRP.csv"
test_feats_path  = r"/home/c/choton/beemachine/codes/AG_vision_2026/dim_reduction/padc_cub_dim_search_logs/cub_reduced_test_GRP.csv"

train_feats_df = load_shape_features(train_feats_path)
val_feats_df = load_shape_features(val_feats_path)
test_feats_df = load_shape_features(test_feats_path)

train_feats_df

,GRP_comp_0,GRP_comp_1,GRP_comp_2,GRP_comp_3,GRP_comp_4,GRP_comp_5,GRP_comp_6,GRP_comp_7,GRP_comp_8,GRP_comp_9,...,GRP_comp_1980,GRP_comp_1981,GRP_comp_1982,GRP_comp_1983,GRP_comp_1984,GRP_comp_1985,GRP_comp_1986,GRP_comp_1987,GRP_comp_1988,label
0,0.180087,-2.501359,-1.721707,0.672200,-1.737625,0.405755,-1.453522,-0.041040,-1.151723,-1.169184,...,-0.856559,1.204895,-1.017067,0.494755,0.048302,1.351737,-0.950983,-2.138452,-1.620418,83.0
1,-1.396131,-0.663380,1.798135,1.293652,0.460872,-2.935147,-0.355554,0.067939,-0.834734,-0.060427,...,-0.647201,0.014706,-0.335415,2.540004,-0.412928,-0.099605,-0.285268,1.628881,-1.022168,69.0
2,1.127864,0.548983,-2.114127,-0.117414,-0.790483,-0.225691,1.504821,-1.736351,-0.026042,0.851543,...,-0.124231,-0.901085,1.397132,-0.670195,0.300474,-0.862046,-0.244936,2.789388,-0.690782,31.0
3,-1.341765,-0.658075,-1.214938,-1.513559,-0.470020,1.499577,0.164688,0.312791,1.651295,0.281758,...,-0.760450,1.327404,0.314096,-0.446304,-1.490587,-0.919376,-0.018027,-0.683104,-0.375539,111.0
4,-0.331188,-1.535139,-0.669717,0.070204,0.200608,1.743427,2.735218,-0.509243,1.487170,-0.173022,...,-0.901131,1.445297,-0.253907,-0.142472,-0.064405,0.227448,0.479049,0.147005,0.009056,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8836,-0.422858,-1.684801,-0.103686,-0.422147,0.036623,-0.438028,0.370486,1.203920,0.553518,-0.894079,...,-1.766663,0.994522,0.157061,-0.012330,-0.712355,0.746043,-0.137047,-0.065007,-0.447921,50.0
8837,-1.011390,-1.796159,1.192450,1.265048,-0.297508,0.897588,0.382126,-0.241616,-0.755979,0.435635,...,-1.022038,-1.195649,1.976699,-0.206485,0.532842,0.420396,0.040910,-0.211689,0.923551,195.0
8838,1.246327,1.189077,-0.579539,0.470667,1.411323,0.212360,-0.432643,-1.283641,0.817287,0.656277,...,0.191983,-0.621141,-0.851488,0.494719,-0.454076,1.502580,-0.984066,1.174095,0.024333,10.0
8839,1.256487,1.604190,-0.417145,-1.004271,-0.539249,-0.594348,-0.066861,1.195046,-0.989282,-0.732909,...,0.815957,-1.482518,1.705745,0.534874,1.219758,-0.923213,0.198530,-1.268983,0.431321,195.0


In [10]:
# Get the 'image' column for joining
train_df['image'] = train_df['filepath'].str.split('/').str[1]
val_df['image'] = val_df['filepath'].str.split('/').str[1]
test_df['image'] = test_df['filepath'].str.split('/').str[1]

# join the datasets
train_df = pd.merge(train_df, train_feats_df, on='label', how='left')
val_df = pd.merge(val_df, train_feats_df, on='label', how='left')
test_df = pd.merge(test_df, train_feats_df, on='label', how='left')

train_df

/tmp/ipykernel_1462820/1234848380.py:7: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  train_df = pd.merge(train_df, train_feats_df, on='label', how='left')
/tmp/ipykernel_1462820/1234848380.py:8: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  val_df = pd.merge(val_df, train_feats_df, on='label', how='left')
/tmp/ipykernel_1462820/1234848380.py:9: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  test_df = pd.merge(test_df, train_feats_df, on='label', how='left')


,img_id,filepath,label,is_train,image,GRP_comp_0,GRP_comp_1,GRP_comp_2,GRP_comp_3,GRP_comp_4,...,GRP_comp_1979,GRP_comp_1980,GRP_comp_1981,GRP_comp_1982,GRP_comp_1983,GRP_comp_1984,GRP_comp_1985,GRP_comp_1986,GRP_comp_1987,GRP_comp_1988
0,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,0.180087,-2.501359,-1.721707,0.672200,-1.737625,...,1.462696,-0.856559,1.204895,-1.017067,0.494755,0.048302,1.351737,-0.950983,-2.138452,-1.620418
1,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,-0.784825,-3.673621,-1.166043,-0.366971,0.369344,...,0.343346,-0.479682,0.355011,-1.262930,0.431965,1.571633,1.030362,2.315140,1.846038,0.232270
2,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,-0.702487,-2.388425,-1.216051,1.129431,-1.493092,...,1.454925,-1.378412,0.716574,-1.597736,-1.179531,1.106610,1.132939,1.337252,-0.132754,-0.162277
3,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,-1.683235,-3.698220,0.935478,-0.350325,-1.036212,...,-0.582025,0.142910,1.608878,1.781450,-0.523226,-0.123844,0.992724,1.108937,-0.453660,1.015911
4,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,-0.204474,-2.087417,-0.063699,0.847168,-1.452911,...,-1.132112,-1.285725,-1.053705,-2.401592,2.368814,0.181567,-0.195902,1.542806,-0.750291,0.730116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389738,860,016.Painted_Bunting/Painted_Bunting_0005_15202...,15,1,Painted_Bunting_0005_15202.jpg,0.407253,1.027648,0.294004,-0.531978,0.793166,...,1.940802,-1.085614,-1.303586,-0.145927,0.778786,-1.580007,-0.219505,-0.669085,0.994789,-0.378255
389739,860,016.Painted_Bunting/Painted_Bunting_0005_15202...,15,1,Painted_Bunting_0005_15202.jpg,0.468591,1.340360,1.709316,-0.759518,0.630343,...,1.157497,-0.109421,-1.265727,-2.016912,1.185627,0.522807,-0.668902,-0.561663,0.936764,-0.781733
389740,860,016.Painted_Bunting/Painted_Bunting_0005_15202...,15,1,Painted_Bunting_0005_15202.jpg,2.202037,1.250118,1.556775,-0.609599,-0.308131,...,-0.824423,-0.323429,-0.372573,-2.104611,0.063039,0.318068,-1.058914,-2.764310,-1.618625,-0.246811
389741,860,016.Painted_Bunting/Painted_Bunting_0005_15202...,15,1,Painted_Bunting_0005_15202.jpg,1.341985,0.595686,-0.232215,0.028078,0.104418,...,1.830219,0.755342,-0.937623,0.504850,-0.057441,-0.515882,0.621734,-0.299538,0.612119,-0.325441


In [11]:
val_df

,img_id,filepath,label,is_train,image,GRP_comp_0,GRP_comp_1,GRP_comp_2,GRP_comp_3,GRP_comp_4,...,GRP_comp_1979,GRP_comp_1980,GRP_comp_1981,GRP_comp_1982,GRP_comp_1983,GRP_comp_1984,GRP_comp_1985,GRP_comp_1986,GRP_comp_1987,GRP_comp_1988
0,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,-1.707567,1.112482,0.564237,0.473420,-0.586084,...,-1.596101,0.370044,2.286559,1.476883,-1.297820,-0.553786,3.153472,0.232161,4.733514,0.278930
1,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,-0.291967,0.952254,-0.786955,-1.376877,0.287331,...,0.795768,-2.286897,-0.062544,-0.340460,-2.602368,-0.790406,-0.871933,-0.617553,0.170748,-0.626206
2,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,-0.926722,-0.306942,-1.253662,1.002596,1.145924,...,0.417506,-1.027391,0.469045,0.341282,-1.134527,-0.081382,-0.604079,0.138408,1.666536,-0.332214
3,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,-0.251917,-0.934641,-2.988021,-0.307872,1.887922,...,-1.816921,-2.695543,-0.431598,0.196073,-0.310413,-1.367475,1.473451,-0.642123,0.430526,-1.419528
4,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,0.065147,-0.548095,-0.407400,-0.112461,-0.913397,...,-1.923743,-0.261197,-0.276387,-0.549346,-1.584526,1.788430,-0.997984,-1.458556,1.186136,-1.197509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77948,8552,146.Forsters_Tern/Forsters_Tern_0082_151937.jpg,145,0,Forsters_Tern_0082_151937.jpg,-0.771537,0.755640,1.044428,0.077041,0.888997,...,-0.500927,-0.320370,0.647907,-1.039626,2.621963,2.106248,-2.085690,-0.549772,-0.405035,-1.201032
77949,8552,146.Forsters_Tern/Forsters_Tern_0082_151937.jpg,145,0,Forsters_Tern_0082_151937.jpg,-1.306618,-1.456416,-0.620347,0.343936,1.246813,...,-0.129164,0.788202,2.243142,0.281728,0.555922,-1.501534,0.941058,0.313679,-4.296170,-0.294577
77950,8552,146.Forsters_Tern/Forsters_Tern_0082_151937.jpg,145,0,Forsters_Tern_0082_151937.jpg,0.532903,-3.439267,1.679508,1.363381,-0.163208,...,-1.041591,-1.065373,3.101680,-2.154768,-2.265839,2.312847,-1.073605,-0.490212,-2.892923,-1.233901
77951,8552,146.Forsters_Tern/Forsters_Tern_0082_151937.jpg,145,0,Forsters_Tern_0082_151937.jpg,-0.339203,-1.970689,1.427266,1.967577,-0.695798,...,-0.964373,-0.988532,1.760420,-0.640216,0.938656,0.047487,-1.028339,0.219967,-1.681967,0.328935


In [12]:
test_df

,img_id,filepath,label,is_train,image,GRP_comp_0,GRP_comp_1,GRP_comp_2,GRP_comp_3,GRP_comp_4,...,GRP_comp_1979,GRP_comp_1980,GRP_comp_1981,GRP_comp_1982,GRP_comp_1983,GRP_comp_1984,GRP_comp_1985,GRP_comp_1986,GRP_comp_1987,GRP_comp_1988
0,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,-1.964664,1.528594,0.114513,-1.054997,-0.499856,...,1.120441,-0.366183,0.249961,0.454113,-0.121026,0.254035,2.504937,-0.391667,0.183229,-1.309148
1,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,-0.077208,-0.572474,-0.671215,-1.219756,-0.235799,...,2.125240,1.765022,-0.058203,-3.598343,2.074750,1.170104,-0.630623,0.986130,-0.734942,0.075113
2,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,-0.997438,-2.259390,0.285245,1.064039,-0.597019,...,1.269936,-0.319995,-0.138921,0.725499,0.309711,0.708952,0.822031,-0.148352,-1.017544,0.112140
3,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,0.583815,-1.595075,1.363108,-0.031111,0.946165,...,-0.461339,-0.846664,0.664379,0.285746,-1.014011,-0.102980,0.154735,-1.969712,-2.186314,-1.243679
4,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,-2.094022,0.313312,-2.182404,1.354540,0.340496,...,1.084857,-1.795825,0.999940,2.688505,1.538431,0.780675,0.217373,-0.670214,0.714756,-1.860241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51984,5432,093.Clark_Nutcracker/Clark_Nutcracker_0066_853...,92,1,Clark_Nutcracker_0066_85390.jpg,0.725079,0.862299,1.503657,-0.431794,-0.615485,...,0.251333,1.901536,0.012266,-0.159681,-0.569272,-1.261629,-0.059121,-0.355972,-2.052960,-0.168922
51985,5432,093.Clark_Nutcracker/Clark_Nutcracker_0066_853...,92,1,Clark_Nutcracker_0066_85390.jpg,-1.782947,0.670579,-0.334058,-0.528952,-0.602503,...,1.166322,0.845934,0.003549,-1.280867,0.656920,-0.017106,1.405328,-0.536392,0.807053,-0.269913
51986,5432,093.Clark_Nutcracker/Clark_Nutcracker_0066_853...,92,1,Clark_Nutcracker_0066_85390.jpg,-0.022631,0.585320,-0.309505,-0.976480,-1.112874,...,2.765462,-0.271311,0.169589,-0.823838,0.155058,-1.708527,-0.152015,1.053616,0.203026,-0.796031
51987,5432,093.Clark_Nutcracker/Clark_Nutcracker_0066_853...,92,1,Clark_Nutcracker_0066_85390.jpg,-0.826766,1.772294,-0.660816,-0.561744,0.657574,...,-0.191733,-0.556216,2.005714,-1.192154,2.018768,-0.963657,1.094850,-0.303645,-1.750160,0.386005


In [13]:
# ======================================================
# Helper: Load shape features
# ======================================================
def extract_shape_features(df):
    df = df.drop(columns=["image", "img_id", "filepath", "label", "is_train"])
    df = df.fillna(df.mean())
    # labels = torch.tensor(df["label"].values, dtype=torch.long)
    # Drop non-feature columns: assuming CSV columns are ['image', 'label', ...features]
    feats = df.to_numpy(dtype=np.float32)
    feats_tensor = torch.tensor(feats, dtype=torch.float32)
    return feats_tensor #, labels

In [14]:
train_feats = extract_shape_features(train_df)
val_feats = extract_shape_features(val_df)
test_feats = extract_shape_features(test_df)

train_mean = train_feats.mean(dim=0)
train_std = train_feats.std(dim=0) + 1e-6

train_feats = (train_feats - train_mean) / train_std
val_feats = (val_feats - train_mean) / train_std
test_feats = (test_feats - train_mean) / train_std

FEATURE_SIZE = train_feats.shape[1]
print(f"Train feats: {train_feats.shape}, Val feats: {val_feats.shape}, Test feats: {test_feats.shape}")
print(f"number of shape features:", FEATURE_SIZE)

Train feats: torch.Size([389743, 1989]), Val feats: torch.Size([77953, 1989]), Test feats: torch.Size([51989, 1989])
number of shape features: 1989


In [15]:
class ShapeEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, embed_dim)

    def forward(self, mask_tensor):
        """
        mask_tensor: (B, 3, 64, 64)
        """
        feat = self.encoder(mask_tensor)
        feat = feat.flatten(1)
        z = self.fc(feat)
        return z  

class GatedFusion(nn.Module):
    def __init__(self, img_dim, shape_dim):
        super().__init__()

        self.shape_proj = nn.Linear(shape_dim, img_dim)

        self.gate = nn.Sequential(
            nn.Linear(img_dim + img_dim, img_dim),
            nn.LayerNorm(img_dim),
            nn.Sigmoid()
        )

    def forward(self, z_img, z_shape):
        z_shape_proj = self.shape_proj(z_shape)

        concat = torch.cat([z_img, z_shape_proj], dim=1)
        g = self.gate(concat)

        fused = z_img + g * (z_shape_proj - z_img)
        return fused

class DeepShapeFusionModel(nn.Module):
    def __init__(self, num_classes, shape_embed_dim=256):
        super().__init__()
        self.num_shape_feats = FEATURE_SIZE

        # Modules
        self.seg_module = load_segmentation(
            name=SEGMENTATION_NAME, 
            encoder_name=SEGMENTATION_ENCODER, 
            checkpoint_path=SEGMENTATION_WEIGHTS
        )
        self.backbone = timm.create_model(model_name=CLASSIFIER_NAME, pretrained=True, num_classes=0)
        self.shape_encoder = ShapeEncoder(shape_embed_dim)

        self.fusion = GatedFusion(
            img_dim=self.backbone.num_features,
            shape_dim=shape_embed_dim
        )

        total_feats = self.backbone.num_features + self.num_shape_feats # + shape_embed_dim

        self.classifier = nn.Sequential(
                                nn.LayerNorm(total_feats),
                                nn.Dropout(0.3),
                                nn.Linear(total_feats, num_classes)
                            )

        for p in self.seg_module.parameters():
            p.requires_grad = False
        self.seg_module.eval()

    def forward(self, x, shape_feats=None):

        # ---- Segmentation ----
        with torch.no_grad():
            mask_logits = self.seg_module(x)
            masks = torch.argmax(mask_logits, dim=1)  # (B, H, W) with labels 0,1,2,3
        # mask_logits = self.seg_module(x)
        # mask_probs = torch.softmax(mask_logits, dim=1)  # example single part

        # Extract part embeddings
        parts = mask_logits[:, 1:, :, :]  # remove background, will try mask_probs as well
        mask_binary = parts.sum(dim=1, keepdim=True)
        edge = torch.abs(
            torch.nn.functional.avg_pool2d(mask_binary, 3, stride=1, padding=1)
            - mask_binary
        )
        shape_tensor = torch.cat([mask_binary, edge, mask_binary], dim=1)
        z_shape = self.shape_encoder(shape_tensor)

        # ---- Extract hand-crafted shape descriptors ----
        # ---- Shape features ----
        if shape_feats is None:
            batch_size = x.size(0)
            batch_feats = []
            for i in range(batch_size):
                img_np = x[i].detach().cpu().numpy().transpose(1, 2, 0)
                mask_np = masks[i].detach().cpu().numpy()
                feats_dict = extract_all_features(img_np, mask_np)
                # Convert dict to tensor (relies on consistent insertion order for 937 features)
                feat_tensor = torch.tensor(list(feats_dict.values()), dtype=torch.float32)
                batch_feats.append(feat_tensor)
            shape_feats = torch.stack(batch_feats).to(x.device)  # (B, 937)
        # shape_feats = torch.tensor(shape_feats).to(x.device)        

        # ---- Image Encoding ----
        z_img = self.backbone(x)
        fused = self.fusion(z_img, z_shape)
        combined = torch.cat([fused, shape_feats], dim=1)  # (B, total_feats)

        # ---- Classification ----
        out = self.classifier(combined)
        return out, mask_logits, z_shape


In [16]:
device = torch.device(f"cuda:{DEVICE_ID}")
model = DeepShapeFusionModel(num_classes=num_classes, shape_embed_dim=512)
model.to(device)

DeepShapeFusionModel(
  (seg_module): DeepLabV3Plus(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_runni

In [17]:
# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
scaler = torch.amp.GradScaler('cuda')

In [18]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    b_pos = 0 # Batch position
    for batch_idx, (images, labels, indices) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)        
        # shape_feats_batch = train_feats[b_pos:b_pos+images.size(0)].to(device)
        shape_feats_batch = train_feats[indices].to(device)
        b_pos += images.size(0)

        optimizer.zero_grad()
        outputs, _, _ = model(images, shape_feats_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    b_pos = 0 # Batch position
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for batch_idx, (images, labels, indices) in enumerate(pbar):
            images, labels = images.to(device), labels.to(device)
            # shape_feats_batch = val_feats[b_pos:b_pos+images.size(0)].to(device)
            shape_feats_batch = val_feats[indices].to(device)
            b_pos += images.size(0)

            # optimizer.zero_grad()
            outputs, _, _ = model(images, shape_feats_batch)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [19]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


 Epoch 1/100 | Train Loss: 4.8269 | Train Acc: 0.0735 | Val Loss: 3.4230 | Val Acc: 0.2919


 Epoch 2/100 | Train Loss: 2.5123 | Train Acc: 0.5365 | Val Loss: 2.1605 | Val Acc: 0.5877


 Epoch 3/100 | Train Loss: 1.6184 | Train Acc: 0.8119 | Val Loss: 1.8522 | Val Acc: 0.6940


 Epoch 4/100 | Train Loss: 1.2878 | Train Acc: 0.9112 | Val Loss: 1.7406 | Val Acc: 0.7342


...

 Epoch 6/100 | Train Loss: 1.0500 | Train Acc: 0.9841 | Val Loss: 1.7076 | Val Acc: 0.7687


 Epoch 7/100 | Train Loss: 1.0055 | Train Acc: 0.9917 | Val Loss: 1.6888 | Val Acc: 0.7704


 Epoch 8/100 | Train Loss: 0.9777 | Train Acc: 0.9947 | Val Loss: 1.6833 | Val Acc: 0.7828


 Epoch 9/100 | Train Loss: 0.9650 | Train Acc: 0.9958 | Val Loss: 1.6926 | Val Acc: 0.7817


KeyboardInterrupt: 

In [20]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.6958 | Test Acc: 0.7735
